# Librerías

In [22]:
!pip install pandas scikit-learn matplotlib

In [23]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Carga de datos

In [24]:
!wget http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

--2025-09-17 10:53:22--  http://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-100k.zip [following]
--2025-09-17 10:53:22--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip.1’

ml-100k.zip.1       100%[===================>]   4.70M  9.97MB/s    in 0.5s    

2025-09-17 10:53:23 (9.97 MB/s) - ‘ml-100k.zip.1’ saved [4924029/4924029]

Archive:  ml-100k.zip
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data      

In [25]:
# Cargo ratings
ratings = pd.read_csv("ml-100k/u.data", sep="\t",
                      names=["user_id", "movie_id", "rating", "timestamp"])

# Cargo movies
movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",  # lo pongo para los títulos con caracteres especiales
    names=[
        "movie_id", "title", "release_date", "video_release_date", "IMDb_URL"
    ] + [f"genre_{i}" for i in range(19)]  # esto son los 19 posibles géneros
)

In [26]:
# Printeo los datos de ratings
print(ratings.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [27]:
# Printeo los datos de movies
print(movies.head())

   movie_id              title release_date  video_release_date  \
0         1   Toy Story (1995)  01-Jan-1995                 NaN   
1         2   GoldenEye (1995)  01-Jan-1995                 NaN   
2         3  Four Rooms (1995)  01-Jan-1995                 NaN   
3         4  Get Shorty (1995)  01-Jan-1995                 NaN   
4         5     Copycat (1995)  01-Jan-1995                 NaN   

                                            IMDb_URL  genre_0  genre_1  \
0  http://us.imdb.com/M/title-exact?Toy%20Story%2...        0        0   
1  http://us.imdb.com/M/title-exact?GoldenEye%20(...        0        1   
2  http://us.imdb.com/M/title-exact?Four%20Rooms%...        0        0   
3  http://us.imdb.com/M/title-exact?Get%20Shorty%...        0        1   
4  http://us.imdb.com/M/title-exact?Copycat%20(1995)        0        0   

   genre_2  genre_3  genre_4  ...  genre_9  genre_10  genre_11  genre_12  \
0        0        1        1  ...        0         0         0         0   


## Ahora veo los nulos

In [28]:
# Reviso valores nulos
print(ratings.isnull().sum())

user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64


In [29]:
# Revisar valores nulos
print(movies.isnull().sum())

movie_id                 0
title                    0
release_date             1
video_release_date    1682
IMDb_URL                 3
genre_0                  0
genre_1                  0
genre_2                  0
genre_3                  0
genre_4                  0
genre_5                  0
genre_6                  0
genre_7                  0
genre_8                  0
genre_9                  0
genre_10                 0
genre_11                 0
genre_12                 0
genre_13                 0
genre_14                 0
genre_15                 0
genre_16                 0
genre_17                 0
genre_18                 0
dtype: int64


**Veo que en la columna `video_release_date` hay demasiados nulos así que la elimino**

In [30]:
#Elimino la columna
movies = movies.drop('video_release_date', axis=1)

Relleno las que tienen pocos nulls

In [31]:
movies['IMDb_URL'] = movies['IMDb_URL'].fillna('Unknown', inplace=True)
movies['release_date'] = movies['release_date'].fillna('Unknown', inplace=True)

/tmp/ipython-input-4238056327.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movies['IMDb_URL'] = movies['IMDb_URL'].fillna('Unknown', inplace=True)
/tmp/ipython-input-4238056327.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value,

## Defino los formatos

In [32]:
ratings['user_id'] = ratings['user_id'].astype(int)
ratings['movie_id'] = ratings['movie_id'].astype(int)
# He puesto float por si se quieren poner ratings que no sean exactos
ratings['rating'] = ratings['rating'].astype(float)

# Construcción de la Matriz de Calificaciones

Me creo la matriz de ratings

In [33]:
ratings_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)
print(ratings_matrix.head())

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   2.0  ...   
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5          4.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
5          0.0   0.0   0.0   0.0  

**Aquí tengo por lo tanto una matriz en la que en cada fila se ve las películas puntuadas por cada uno de los usuarios, es decir, cada fila corresponde a un usuario.**

# Cálculo de la Similitud entre Películas:

In [34]:
# Hago la traspuesta para que cada fila contenga las valoraciones de todos los usuarios para una película
# Después de hacer la traspuesta calculo la similitud entre todos los vectores de valoraciones que quedan de las películas con el `cosine_similarity`
cosine_sim = cosine_similarity(ratings_matrix.T)
# Creo el df haciendo que tanto los index como las columnas sean los ids de las películas
cosine_sim_df = pd.DataFrame(cosine_sim, index=ratings_matrix.columns, columns=ratings_matrix.columns)
print(cosine_sim_df.head())

movie_id      1         2         3         4         5         6     \
movie_id                                                               
1         1.000000  0.402382  0.330245  0.454938  0.286714  0.116344   
2         0.402382  1.000000  0.273069  0.502571  0.318836  0.083563   
3         0.330245  0.273069  1.000000  0.324866  0.212957  0.106722   
4         0.454938  0.502571  0.324866  1.000000  0.334239  0.090308   
5         0.286714  0.318836  0.212957  0.334239  1.000000  0.037299   

movie_id      7         8         9         10    ...      1673  1674  \
movie_id                                          ...                   
1         0.620979  0.481114  0.496288  0.273935  ...  0.035387   0.0   
2         0.383403  0.337002  0.255252  0.171082  ...  0.000000   0.0   
3         0.372921  0.200794  0.273669  0.158104  ...  0.000000   0.0   
4         0.489283  0.490236  0.419044  0.252561  ...  0.000000   0.0   
5         0.334769  0.259161  0.272448  0.055453  ...  0.

# Desarrollo la función para retornar las películas más similares

In [35]:
def n_peliculas_similares(movie_title, cosine_sim_df, N, movies):
  # Consigo con el df movies que obtuve al principio coger el id a partir del título de la película
  # Esto lo hago localizando la fila que contiene el título y devolviendo el valor de la columna 'movie_id' en esa fila
  movie_id = movies.loc[movies['title'] == movie_title, 'movie_id'].values[0]
  # A partir de la matriz de similitud devuelvo de mayor a menor los valores de la columna que tenga el id de la película
  # Devuelvo a partir del segundo valor para no devolver la propia película que tiene 1.0 consigo misma
  similares_ID = cosine_sim_df[movie_id].sort_values(ascending=False).iloc[1:N+1]
  # Mapeo a títulos y cojo sus puntuaciones de similitud también
  # como lo hago dentro de una list comprehension obtengo una lista de tuplas
  similares = [(movies.loc[movies['movie_id'] == movieID, 'title'].values[0], sim) for movieID, sim in similares_ID.items()]
  return similares

## Pongo una función extra para devolver por pantalla las películas que hay dentro de la lista

In [40]:
def printear_lista(lista):
  print(f"Top {len(lista)} de películas:")
  # Recorro cada pelicula y su puntuación de similitud y empiezo desde el 1
  for i, (pelicula, score_sim) in enumerate(lista, 1):
    print(f"{i}. {pelicula} --> similitud: {score_sim:.3f}")

# Ejemplos de uso

In [41]:
top_5_peliculas = n_peliculas_similares('GoldenEye (1995)', cosine_sim_df, 5, movies)
printear_lista(top_5_peliculas)

Top 5 de películas:
1. Under Siege (1992) --> similitud: 0.660
2. Top Gun (1986) --> similitud: 0.624
3. True Lies (1994) --> similitud: 0.617
4. Batman (1989) --> similitud: 0.616
5. Stargate (1994) --> similitud: 0.605


In [42]:
top_5_peliculas = n_peliculas_similares('Under Siege (1992)', cosine_sim_df, 5, movies)
printear_lista(top_5_peliculas)

Top 5 de películas:
1. Die Hard 2 (1990) --> similitud: 0.700
2. GoldenEye (1995) --> similitud: 0.660
3. Die Hard: With a Vengeance (1995) --> similitud: 0.654
4. Clear and Present Danger (1994) --> similitud: 0.633
5. True Lies (1994) --> similitud: 0.627


In [43]:
top_10_peliculas = n_peliculas_similares('GoldenEye (1995)', cosine_sim_df, 10, movies)
printear_lista(top_10_peliculas)

Top 10 de películas:
1. Under Siege (1992) --> similitud: 0.660
2. Top Gun (1986) --> similitud: 0.624
3. True Lies (1994) --> similitud: 0.617
4. Batman (1989) --> similitud: 0.616
5. Stargate (1994) --> similitud: 0.605
6. Cliffhanger (1993) --> similitud: 0.602
7. Die Hard 2 (1990) --> similitud: 0.597
8. Batman Returns (1992) --> similitud: 0.596
9. Die Hard: With a Vengeance (1995) --> similitud: 0.590
10. Terminator 2: Judgment Day (1991) --> similitud: 0.584
